In [1]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

from evedesign.system import System, Protein
from evedesign.models.boltzfold import BoltzFoldTransformer
from evedesign.models.mpnn import LigandMPNN

/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[15:22:23] Initializing Normalizer


In [3]:
system = System([
    Protein(
        id="EcCM", rep="TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH", first_index=2
    ),
])
print(f"Sequence length: {len(system[0].rep)}")

Sequence length: 94


# 1. Fold with Boltz, to be replaced with a generated sequence from BoltzGen

In [4]:
# Alternatively, you can use 
# # retrieve and add evolutionary sequences to system
# system = add_sequences_mmseqs2(
#     system, use_pairing=False, use_env=True
# )

# # perform some extra redundancy reduction on sequences which ColabFold server does not handle with env=True
# system[0].sequences = filter_entity_sequences_mmseqs(
#     system[0], max_seq_id=0.90
# )
# boltz = BoltzFoldTransformer(
#     device='cpu',
#     use_msa=True,
#     diffusion_samples=1,
# )

from evedesign.tools.mmseqs2 import add_sequences_mmseqs2

system = add_sequences_mmseqs2(system, use_pairing=True)

boltz = BoltzFoldTransformer(
    device='cpu',
    use_msa=True,
    diffusion_samples=1,
)

boltz.build(system)
folded = boltz.transform([system.rep_to_instance()])

print(f"Score: {folded[0].score}")
print(f"Confidence (pLDDT): {folded[0].confidence}")

Processing 1 inputs with 1 threads.


  0%|          | 0/1 [00:00<?, ?it/s]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_4mucpl2r/inputs/instance_0.yaml with 1 protein entities.
Calling MSA server for target instance_0 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


100%|██████████| 1/1 [00:05<00:00,  5.22s/it]
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-04-14 15:25:36.796 | INFO     | evedesign.models.boltzfold:_load_model:206 - Boltz-2 loaded from /Users/khbelahsen/.boltz/boltz2_conf.ckpt
2026-04-14 15:34:54.525 | INFO     | evedesign.models.boltzfold:transform:369 - Boltz-2 output written to: /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_4mucpl2r/predictions
2026-04-14 15:34:54.566 | INFO     | evedesign.models.boltzfold:transform:370 - Files written (6):
2026-04-14 15:34:54.570 | INFO     | evedesign.models.boltzfold:transform:373 -   instance_0/confidence_instance_0_model_0.json (443 bytes)
2026-04-14 15:34:54.573 | INFO     | evedesign.models.boltzfold:transform:373 -   instance_0/instance_0_model_0.cif (67

Score: 0.9227787852287292
Confidence (pLDDT): 0.9394010305404663


In [5]:
ei = folded[0][0]
chain_id = list(ei.models.keys())[0]
structure = ei.models[chain_id]
print(f"Chain: {chain_id}")
print(f"Atom count: {len(structure.atom_array)}")

Chain: A
Atom count: 762


In [6]:
!pip install py3Dmol -q
import io
import py3Dmol

buf = io.StringIO()
structure.to_file(buf, format="cif")

view = py3Dmol.view(width=800, height=400)
view.addModel(buf.getvalue(), "cif")
view.setStyle({
    "cartoon": {
        "colorscheme": {
            "prop": "b",
            "gradient": "roygb",
            "min": 50,
            "max": 90,
        }
    }
})
view.zoomTo()
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [7]:
s_designable = system.apply_instance(folded[0])
print(f"Structures attached: {s_designable[0].structures is not None}")
print(f"Structure keys: {list(s_designable[0].structures.keys())}")

Structures attached: True
Structure keys: ['A']


# 2. Run MPNN sequence optimisation

In [ ]:
mpnn = LigandMPNN(model_name='proteinmpnn_v_48_020')
mpnn.build(s_designable)
designs = mpnn.generate(num_designs=5)

print(f"Generated {len(designs)} designs")
for i, d in enumerate(designs):
    seq_designed = "".join(d[0].rep)
    print(f"  Design {i}: score={d.score:.3f} seq={seq_designed[:30]}...")

2026-04-14 15:35:40.516 | INFO     | evedesign.models.mpnn:download_checkpoint:98 - Using cached checkpoint at ./model_params/proteinmpnn_v_48_020.pt


Generated 5 designs
  Design 0: score=0.919 seq=GAADPAEAAAAALAAAAAEAAAALAAAAEA...
  Design 1: score=0.865 seq=GALDPAALAAAAAAAAAALAAAALAAALAA...
  Design 2: score=0.917 seq=GALDPAALAAAAAAAAAAALAAALAAALEA...
  Design 3: score=0.857 seq=GPLDPAALAAAAAAAAAAALAALLAAALAA...
  Design 4: score=0.944 seq=AALDPAALAAAAAAAAAAALAALLAAALAA...


In [9]:
designs

[SystemInstance([EntityInstance(rep=GAADPAEAAAAALAAAAAEAAAALAAAAEAARAAAAAKLAAGKPVADPEA..., models=None)] id=None score=0.9186083078384399),
 SystemInstance([EntityInstance(rep=GALDPAALAAAAAAAAAALAAAALAAALAAAEAVGRAKIAAGLPVVDPAA..., models=None)] id=None score=0.865038275718689),
 SystemInstance([EntityInstance(rep=GALDPAALAAAAAAAAAAALAAALAAALEAARAVAAARLAAGLPVVDPAA..., models=None)] id=None score=0.9174126386642456),
 SystemInstance([EntityInstance(rep=GPLDPAALAAAAAAAAAAALAALLAAALAAARARAEAKLAAGLPVADPAA..., models=None)] id=None score=0.8570456504821777),
 SystemInstance([EntityInstance(rep=AALDPAALAAAAAAAAAAALAALLAAALAAAKAAAAARAAAGLPVVDPAA..., models=None)] id=None score=0.9442359209060669)]

# 3. Refold with Boltz2

In [21]:
s_designable = add_sequences_mmseqs2(s_designable, use_pairing=True)

refolded = BoltzFoldTransformer(
    device='cpu',
    use_msa=True,
    sampling_steps=200,
    diffusion_samples=1,
).build(s_designable).transform(designs)

print(f"Refolded {len(refolded)} designs")
for i, r in enumerate(refolded):
    print(f"  Design {i}: score={r.score:.4f} confidence={r.confidence:.4f}")

2026-04-14 16:47:10.310 | WARNING  | evedesign.models.boltz.convert:_resolve_msa_field:102 - Entity 'EcCM': structures are present but template conditioning is not yet implemented — ignoring.
2026-04-14 16:47:10.316 | WARNING  | evedesign.models.boltz.convert:_resolve_msa_field:102 - Entity 'EcCM': structures are present but template conditioning is not yet implemented — ignoring.
2026-04-14 16:47:10.319 | WARNING  | evedesign.models.boltz.convert:_resolve_msa_field:102 - Entity 'EcCM': structures are present but template conditioning is not yet implemented — ignoring.
2026-04-14 16:47:10.321 | WARNING  | evedesign.models.boltz.convert:_resolve_msa_field:102 - Entity 'EcCM': structures are present but template conditioning is not yet implemented — ignoring.
2026-04-14 16:47:10.323 | WARNING  | evedesign.models.boltz.convert:_resolve_msa_field:102 - Entity 'EcCM': structures are present but template conditioning is not yet implemented — ignoring.


Processing 5 inputs with 1 threads.


  0%|          | 0/5 [00:00<?, ?it/s]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_k0zngjy9/inputs/instance_0.yaml with 1 protein entities.
Calling MSA server for target instance_0 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


 20%|██        | 1/5 [00:02<00:11,  2.78s/it]

...

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_k0zngjy9/inputs/instance_3.yaml with 1 protein entities.
Calling MSA server for target instance_3 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


 80%|████████  | 4/5 [00:10<00:02,  2.71s/it]

Generating MSA for /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_k0zngjy9/inputs/instance_4.yaml with 1 protein entities.
Calling MSA server for target instance_4 with 1 sequences
MSA server URL: https://api.colabfold.com
MSA pairing strategy: greedy
No authentication provided for MSA server


100%|██████████| 5/5 [00:13<00:00,  2.73s/it]
/Users/khbelahsen/Documents/GitHub/work/marks/evedesign/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2026-04-14 16:47:49.712 | INFO     | evedesign.models.boltzfold:_load_model:206 - Boltz-2 loaded from /Users/khbelahsen/.boltz/boltz2_conf.ckpt
2026-04-14 22:25:23.996 | INFO     | evedesign.models.boltzfold:transform:369 - Boltz-2 output written to: /var/folders/xb/vj5zk8_x0bs4xmp72n7_6dbm0000gn/T/boltzfold_k0zngjy9/predictions
2026-04-14 22:25:24.026 | INFO     | evedesign.models.boltzfold:transform:370 - Files written (30):
2026-04-14 22:25:24.028 | INFO     | evedesign.models.boltzfold:transform:373 -   instance_0/confidence_instance_0_model_0.json (443 bytes)
2026-04-14 22:25:24.029 | INFO     | evedesign.models.boltzfold:transform:373 -   instance_0/instance_0_model_0.cif (5

Refolded 5 designs
  Design 0: score=0.6688 confidence=0.7058
  Design 1: score=0.7225 confidence=0.7747
  Design 2: score=0.7395 confidence=0.7871
  Design 3: score=0.7236 confidence=0.7535
  Design 4: score=0.7831 confidence=0.8229


In [22]:
refolded

[SystemInstance([EntityInstance(rep=GAADPAEAAAAALAAAAAEAAAALAAAAEAARAAAAAKLAAGKPVADPEA..., models=1)] id=None score=0.6688438653945923),
 SystemInstance([EntityInstance(rep=GALDPAALAAAAAAAAAALAAAALAAALAAAEAVGRAKIAAGLPVVDPAA..., models=1)] id=None score=0.7224706411361694),
 SystemInstance([EntityInstance(rep=GALDPAALAAAAAAAAAAALAAALAAALEAARAVAAARLAAGLPVVDPAA..., models=1)] id=None score=0.7395095825195312),
 SystemInstance([EntityInstance(rep=GPLDPAALAAAAAAAAAAALAALLAAALAAARARAEAKLAAGLPVADPAA..., models=1)] id=None score=0.7235867381095886),
 SystemInstance([EntityInstance(rep=AALDPAALAAAAAAAAAAALAALLAAALAAAKAAAAARAAAGLPVVDPAA..., models=1)] id=None score=0.7831134796142578)]

In [23]:
best = max(refolded, key=lambda r: r.score)
best_idx = refolded.index(best)
print(f"Best design: {best_idx} (score={best.score:.4f})")

ei_best = best[0]
chain_id = list(ei_best.models.keys())[0]
structure_best = ei_best.models[chain_id]

buf = io.StringIO()
structure_best.to_file(buf, format="cif")

view = py3Dmol.view(width=800, height=400)
view.addModel(buf.getvalue(), "cif")
view.setStyle({
    "cartoon": {
        "colorscheme": {
            "prop": "b",
            "gradient": "roygb",
            "min": 50,
            "max": 90,
        }
    }
})
view.zoomTo()
view.show()

Best design: 4 (score=0.7831)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# 4. Compare to original structure

In [24]:
import pandas as pd

seq_original = "TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH"
seq_best = "".join(best[0].rep)

matches = sum(a == b for a, b in zip(seq_original, seq_best))
identity = matches / len(seq_original) * 100

print(f"Sequence identity to original: {identity:.1f}%")
print(f"Original score:    {folded[0].score:.4f}")
print(f"Best design score: {best.score:.4f}")
print(f"Improvement:       {best.score - folded[0].score:.4f}")

Sequence identity to original: 21.3%
Original score:    0.9228
Best design score: 0.7831
Improvement:       -0.1397


In [25]:
import biotite.structure as struc
import numpy as np

def compute_rmsd(reference_structure, mobile_structure):
    """
    Compute CA-only RMSD between two structures after
    superimposition. Uses only backbone CA atoms for
    a fair comparison of overall fold similarity.
    """
    ref_ca = reference_structure.atom_array[
        reference_structure.atom_array.atom_name == "CA"
    ]
    mob_ca = mobile_structure.atom_array[
        mobile_structure.atom_array.atom_name == "CA"
    ]

    min_len = min(len(ref_ca), len(mob_ca))
    if min_len == 0:
        return None
    ref_ca = ref_ca[:min_len]
    mob_ca = mob_ca[:min_len]

    fitted, _ = struc.superimpose(ref_ca, mob_ca)
    return struc.rmsd(ref_ca, fitted)

mpnn_scores = [d.score for d in designs]

original_structure = folded[0][0].models[
    list(folded[0][0].models.keys())[0]
]

print("RMSD to original fold (CA atoms, after superimposition):")
print(f"{'Design':>8} {'RMSD (Å)':>10} {'Boltz score':>12} {'MPNN score':>12}")
print("-" * 46)

rmsd_values = []
for i, r in enumerate(refolded):
    ei = r[0]
    if ei.models is None:
        print(f"{i:>8} {'failed':>10}")
        rmsd_values.append(None)
        continue
    chain_id = list(ei.models.keys())[0]
    design_structure = ei.models[chain_id]
    rmsd_val = compute_rmsd(original_structure, design_structure)
    rmsd_values.append(rmsd_val)
    print(
        f"{i:>8} {rmsd_val:>10.2f} "
        f"{r.score:>12.4f} "
        f"{mpnn_scores[i]:>12.3f}"
    )

valid = [v for v in rmsd_values if v is not None]
best_rmsd_idx = int(np.argmin(valid))
print(f"\nMost similar to original fold: design {best_rmsd_idx} "
      f"(RMSD={rmsd_values[best_rmsd_idx]:.2f} Å)")
print(f"Best Boltz score:  design {refolded.index(best)} "
      f"(score={best.score:.4f})")


RMSD to original fold (CA atoms, after superimposition):
  Design   RMSD (Å)  Boltz score   MPNN score
----------------------------------------------
       0       4.19       0.6688        0.919
       1       3.55       0.7225        0.865
       2       4.38       0.7395        0.917
       3       3.42       0.7236        0.857
       4       3.56       0.7831        0.944

Most similar to original fold: design 3 (RMSD=3.42 Å)
Best Boltz score:  design 4 (score=0.7831)


In [30]:
import io
from pathlib import Path
from datetime import datetime

# Create timestamped output directory next to the notebook
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = Path("output") / run_id
output_dir.mkdir(parents=True, exist_ok=True)

# Save original fold
original_chain = list(folded[0][0].models.keys())[0]
original_structure = folded[0][0].models[original_chain]
original_path = output_dir / "original_fold.cif"
original_structure.to_file(str(original_path), format="cif")
print(f"Saved original fold: {original_path}")

# Save all refolded designs
design_paths = []
for i, r in enumerate(refolded):
    ei = r[0]
    if ei.models is None:
        print(f"Design {i}: no structure, skipping")
        design_paths.append(None)
        continue
    chain_id = list(ei.models.keys())[0]
    structure = ei.models[chain_id]
    path = output_dir / f"design_{i}_boltz{r.score:.4f}_mpnn{mpnn_scores[i]:.3f}.cif"
    structure.to_file(str(path), format="cif")
    design_paths.append(path)
    print(f"Saved design {i}: {path.name}")

print(f"\nAll files saved to: {output_dir.resolve()}")
print(f"\nTo align in PyMOL, run:")
print(f"  reinitialize")
print(f"  load {original_path.resolve()}, original_fold")
for i, dpath in enumerate(design_paths):
    if dpath is None:
        continue
    print(f"  load {dpath.resolve()}, design_{i}")
for i, dpath in enumerate(design_paths):
    if dpath is None:
        continue
    print(f"  align design_{i}, original_fold")


Saved original fold: output/20260415_091834/original_fold.cif
Saved design 0: design_0_boltz0.6688_mpnn0.919.cif
Saved design 1: design_1_boltz0.7225_mpnn0.865.cif
Saved design 2: design_2_boltz0.7395_mpnn0.917.cif
Saved design 3: design_3_boltz0.7236_mpnn0.857.cif
Saved design 4: design_4_boltz0.7831_mpnn0.944.cif

All files saved to: /Users/khbelahsen/Documents/GitHub/work/marks/evedesign/examples/structure_validated_design/output/20260415_091834

To align in PyMOL, run:
  reinitialize
  load /Users/khbelahsen/Documents/GitHub/work/marks/evedesign/examples/structure_validated_design/output/20260415_091834/original_fold.cif, original_fold
  load /Users/khbelahsen/Documents/GitHub/work/marks/evedesign/examples/structure_validated_design/output/20260415_091834/design_0_boltz0.6688_mpnn0.919.cif, design_0
  load /Users/khbelahsen/Documents/GitHub/work/marks/evedesign/examples/structure_validated_design/output/20260415_091834/design_1_boltz0.7225_mpnn0.865.cif, design_1
  load /Users/khbe